# Workflow vs. Visual estimate agreement analysis

For each year:

- Ordinary least-squares linear regression (Visual = x, Workflow = y)
- Reports slope, intercept, **R²**, and the regression **p-value**
- **Spearman** rank correlation (rho + p-value)
- Scatterplot with the 1:1 line and the fitted regression line

The final cell stitches the four per-year scatterplots into a single 2×2 figure and prints a cross-year summary table.

**Before running:** set the four column-name constants and the four `csv_path` strings to match your real files.

## Setup & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

# --------------------------------------------------------------------------- #
#  Shared configuration                                                        #
# --------------------------------------------------------------------------- #
# Column names in the CSVs. Change once here if they differ.
COL_PLOT     = "Image"        # plot name
COL_HEADS    = "Num_Heads"       # number of heads (not used in the stats, kept for reference)
COL_WORKFLOW = "Workflow"    # workflow (automated) estimate  -> y axis
COL_VISUAL   = "Visual"      # visual (rater) estimate        -> x axis

OUT_DIR = "."                # where PNGs are written

# Holds each year's fitted results so the final cell can redraw them together.
RESULTS = {}

## Helper: Define analysis functions

In [ ]:
# ---------------------------------------------------------------- style ---- #
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.linewidth": 0.8,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.labelsize": 10, "axes.titlesize": 9,
    "savefig.bbox": "tight", "savefig.dpi": 600,
    "figure.constrained_layout.use": True,
})

POINTS = dict(s=18, alpha=0.65, color="#3b6ea5", edgecolor="k", linewidth=0.25)
FIT_C, REF_C = "crimson", "0.55"

LAMBDA = {2024: 0.152,   # (6.22 / 15.97)**2
          2025: 0.336,   # (12.75 / 22.0)**2
          2026: 5.540,   # (13.89 / 5.9)**2
          2023: 2.009}    # no truth data; mean of 2024-2026

def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"= {p:.3f}"


def despine(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


# ------------------------------------------------------ Deming regression --- #
def deming(x, y, lam):
    """Deming (errors-in-both-variables) regression of y on x.

    lam = var(error in x) / var(error in y).  lam = 1 is orthogonal regression.
    Returns (slope, intercept).
    """
    x, y = np.asarray(x, float), np.asarray(y, float)
    mx, my = x.mean(), y.mean()
    sxx = ((x - mx) ** 2).sum()
    syy = ((y - my) ** 2).sum()
    sxy = ((x - mx) * (y - my)).sum()
    num = (syy - lam * sxx) + np.sqrt((syy - lam * sxx) ** 2 + 4 * lam * sxy ** 2)
    slope = num / (2 * sxy)
    return slope, my - slope * mx


def deming_boot(x, y, lam, n_boot=2000, seed=0):
    """Bootstrap percentile CIs for the Deming slope and intercept."""
    rng = np.random.default_rng(seed)
    n = x.size
    out = np.empty((n_boot, 2))
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        out[i] = deming(x[idx], y[idx], lam)
    lo_s, hi_s = np.percentile(out[:, 0], [2.5, 97.5])
    lo_i, hi_i = np.percentile(out[:, 1], [2.5, 97.5])
    return (lo_s, hi_s), (lo_i, hi_i)


# ------------------------------------------------------------ per year ----- #

def _stats_block(res):
    s_lo, s_hi = res["slope_ci"]
    return (
        rf"$r_s$: {res['spearman_rho']:.2f}" f"{stars(res['spearman_p'])}\n"
        rf"$\lambda$ = {res['lam']:.2f}" "\n"
        f"Slope: {res['slope']:.2f} [{s_lo:.2f}, {s_hi:.2f}]\n"
        f"$n$ = {res['n']}"
    )
def analyze_year(year, csv_path, lam=None, ax=None, annotate=True, n_boot=2000):
    """Deming regression + Spearman for one year; optionally draw its panel.

    lam is var(visual error)/var(workflow error); with RMSE 16 vs 12 -> 1.78.
    """
    lam = LAMBDA[year] if lam is None else lam
    df = pd.read_csv(csv_path)
    x = pd.to_numeric(df[COL_VISUAL], errors="coerce")
    y = pd.to_numeric(df[COL_WORKFLOW], errors="coerce")
    mask = x.notna() & y.notna()
    x, y = x[mask].to_numpy(float), y[mask].to_numpy(float)
    n = x.size

    slope, intercept = deming(x, y, lam)
    (s_lo, s_hi), (i_lo, i_hi) = deming_boot(x, y, lam, n_boot=n_boot)
    rho, rho_p = stats.spearmanr(x, y)

    # Agreement summaries (Deming has no meaningful R^2, so report these).
    diff = y - x
    bias, sd = diff.mean(), diff.std(ddof=1)
    rmsd = np.sqrt((diff ** 2).mean())
    ccc = lins_ccc(x, y)
    slope_differs = not (s_lo <= 1.0 <= s_hi)

    print(f"\n===== {year}  (n = {n}) =====")
    print(f"Deming regression (workflow ~ visual, lambda = {lam:.2f}):")
    print(f"    slope      = {slope:.4f}   95% CI [{s_lo:.3f}, {s_hi:.3f}]"
          f"{'  <- differs from 1' if slope_differs else '  (includes 1)'}")
    print(f"    intercept  = {intercept:.4f}   95% CI [{i_lo:.3f}, {i_hi:.3f}]")
    print("Spearman rank correlation:")
    print(f"    r_s        = {rho:.4f}   p {fmt_p(rho_p)}")
    print("Agreement (workflow - visual):")
    print(f"    bias       = {bias:+.3f}    SD = {sd:.3f}    RMSD = {rmsd:.3f}")
    print(f"    Lin's CCC  = {ccc:.4f}")

    res = dict(year=year, n=n, x=x, y=y, lam=lam,
               slope=slope, intercept=intercept,
               slope_ci=(s_lo, s_hi), intercept_ci=(i_lo, i_hi),
               spearman_rho=rho, spearman_p=rho_p,
               bias=bias, sd_diff=sd, rmsd=rmsd, ccc=ccc)
    RESULTS[year] = res

    if ax is not None or annotate:
        created = ax is None
        if created:
            fig, ax = plt.subplots(figsize=(3.4, 3.4))
        _draw_scatter(ax, res, label=str(year), annotate=annotate)
        if created:
            plt.show()
    return res

def stars(p):
    """Conventional significance markers."""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""    

def lins_ccc(x, y):
    """Lin's concordance correlation coefficient."""
    mx, my = x.mean(), y.mean()
    vx = x.var(ddof=0); vy = y.var(ddof=0)
    cov = ((x - mx) * (y - my)).mean()
    return 2 * cov / (vx + vy + (mx - my) ** 2)


def _draw_scatter(ax, res, label, annotate=True, line=None, legend=False,
                  extrapolate=False, show_ci=False):
    """Draw one Deming scatter panel.

    line         : shared [lo, hi] axis limits; computed per-panel if None.
    extrapolate  : if False (default) the fit is drawn only across the observed
                   x-range, so range-restricted years are self-evident.
    show_ci      : shade the bootstrap CI of the fitted line.
    """
    x, y = res["x"], res["y"]

    if line is None:
        lo = float(min(x.min(), y.min()))
        hi = float(max(x.max(), y.max()))
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        line = np.array([lo - pad, hi + pad])

    # 1:1 reference spans the full panel.
    ax.plot(line, line, ls="--", color=REF_C, lw=0.8, zorder=1, label="1:1")
    ax.scatter(x, y, zorder=3, **POINTS)

    # Fit line: observed x-range only unless asked to extrapolate, then clipped
    # to the panel so steep slopes (e.g. 2026) can't blow out the axes.
    x0, x1 = (line[0], line[1]) if extrapolate else (x.min(), x.max())
    xs = np.linspace(x0, x1, 300)
    ys = res["slope"] * xs + res["intercept"]
    keep = (ys >= line[0]) & (ys <= line[1])

    if show_ci and "slope_ci" in res and "intercept_ci" in res:
        s_lo, s_hi = res["slope_ci"]
        i_lo, i_hi = res["intercept_ci"]
        band = np.vstack([s * xs + i for s in (s_lo, s_hi)
                                     for i in (i_lo, i_hi)])
        ax.fill_between(xs[keep], band.min(0)[keep], band.max(0)[keep],
                        color=FIT_C, alpha=0.15, lw=0, zorder=2)

    ax.plot(xs[keep], ys[keep], color=FIT_C, lw=1.4, zorder=4,
            label="Deming fit")

    if annotate:
        ax.text(0.04, 0.96, _stats_block(res), transform=ax.transAxes,
                va="top", ha="left", fontsize=7.5, linespacing=1.4, zorder=5)

    ax.set_title(label, loc="left", fontweight="bold", pad=4)
    ax.set_xlim(line)
    ax.set_ylim(line)
    ax.set_aspect("equal", adjustable="box")
    despine(ax)
    if legend:
        ax.legend(loc="lower right", frameon=False, fontsize=8)


# ----------------------------------------------- four-panel scatter fig ---- #
def combined_figure(years=(2023, 2024, 2025, 2026), out_path=None, save=False):
    have = [y for y in years if y in RESULTS]
    if not have:
        raise ValueError(f"RESULTS empty — run analyze_year() first. "
                         f"Keys: {list(RESULTS)}")

    fig, axes = plt.subplots(2, 2, figsize=(7.0, 7.0), sharex=True, sharey=True)

    allv = np.concatenate([np.r_[RESULTS[y]["x"], RESULTS[y]["y"]] for y in have])
    lo, hi = float(allv.min()), float(allv.max())
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    line = np.array([lo - pad, hi + pad])

    for ax, year, letter in zip(axes.flat, years, "abcdefgh"):
        res = RESULTS.get(year)
        if res is None:
            ax.set_visible(False); continue
        _draw_scatter(ax, res, label=f"({letter}) {year}", line=line)

    fig.supxlabel("Traditional visual FHB index score (%)")
    fig.supylabel("Workflow FHB index score (%)")
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper right", bbox_to_anchor=(0.99, 0.99),
               frameon=False, fontsize=8)

    plt.show()
    if save:
        path = out_path or f"{OUT_DIR}/workflow_vs_visual_all_years.png"
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        fig.savefig(path)
        fig.savefig(os.path.splitext(path)[0] + ".pdf", dpi=600)
    return summary_table()


def summary_table():
    rows = []
    for r in sorted(RESULTS.values(), key=lambda d: d["year"]):
        rows.append({
            "year": r["year"], "n": r["n"],
            "deming_slope": r["slope"], "slope_lo": r["slope_ci"][0],
            "slope_hi": r["slope_ci"][1], "intercept": r["intercept"],
            "spearman_rs": r["spearman_rho"], "spearman_p": r["spearman_p"],
            "bias": r["bias"], "sd_diff": r["sd_diff"],
            "rmsd": r["rmsd"], "ccc": r["ccc"],
        })
    out = pd.DataFrame(rows)
    print("\n===== Summary across years =====")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    return out


## Year 1

In [ ]:
analyze_year(2023, "#path to csv file")

## Year 2

In [ ]:
analyze_year(2024, "#path to csv file")

## Year 3

In [ ]:
analyze_year(2025, "#path to csv file")

## Year 4

In [ ]:
analyze_year(2026, "#path to csv file")

In [ ]:
combined_figure(years=(2023, 2024, 2025, 2026), out_path="#path to output file", save=True)